In [71]:
import numpy as np
import pandas as pd
import yfinance as yf

In [72]:
def fetch_nifty_data(period="5y"):
    ticker = "SBIN.NS"
    df = yf.download(ticker, period=period)
    df.dropna(inplace=True)
    return df

df = fetch_nifty_data()
print(df['Close'])
print(df['Close'].shift(1))


/tmp/ipython-input-292/82567579.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period=period)
[*********************100%***********************]  1 of 1 completed

Ticker          SBIN.NS
Date                   
2021-03-01   364.426910
2021-03-02   363.734894
2021-03-03   373.746368
2021-03-04   365.072815
2021-03-05   354.000214
...                 ...
2026-02-23  1227.800049
2026-02-24  1223.300049
2026-02-25  1200.099976
2026-02-26  1209.500000
2026-02-27  1201.900024

[1237 rows x 1 columns]
Ticker          SBIN.NS
Date                   
2021-03-01          NaN
2021-03-02   364.426910
2021-03-03   363.734894
2021-03-04   373.746368
2021-03-05   365.072815
...                 ...
2026-02-23  1216.099976
2026-02-24  1227.800049
2026-02-25  1223.300049
2026-02-26  1200.099976
2026-02-27  1209.500000

[1237 rows x 1 columns]


In [73]:
import numpy as np

# Calculating Log Returns
df['Returns'] = np.log(df['Close'] / df['Close'].shift(1))
df['Volatility'] = df['Returns'].rolling(20).std()
df.dropna(inplace=True)
df.dropna(inplace=True)
# Dropping the first NaN value (since there's no previous day for the first row)
print(df)

Price             Close         High          Low         Open    Volume  \
Ticker          SBIN.NS      SBIN.NS      SBIN.NS      SBIN.NS   SBIN.NS   
Date                                                                       
2021-03-31   336.145630   339.421291   330.286402   332.454764  38651025   
2021-04-01   342.004883   343.158278   335.038384   339.282886  31883453   
2021-04-05   326.226440   340.666978   322.074216   339.098348  51743981   
2021-04-06   323.458282   329.409816   322.304887   328.210295  44147709   
2021-04-07   330.840027   335.130682   320.736297   324.104207  48023602   
...                 ...          ...          ...          ...       ...   
2026-02-23  1227.800049  1231.099976  1217.099976  1222.000000   9733835   
2026-02-24  1223.300049  1234.699951  1219.699951  1232.000000  15788449   
2026-02-25  1200.099976  1228.099976  1197.500000  1228.099976  19581100   
2026-02-26  1209.500000  1215.400024  1188.900024  1201.000000  16196052   
2026-02-27  

In [74]:
X = df[['Returns', 'Volatility']].values
X

array([[ 0.00965387,  0.01719441],
       [ 0.01728053,  0.01784268],
       [-0.04723327,  0.01892886],
       ...,
       [-0.0191473 ,  0.02281147],
       [ 0.00780218,  0.02279925],
       [-0.00630339,  0.02294884]])

In [75]:
!pip install hmmlearn

In [76]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[['Returns','Volatility']])

In [77]:

from hmmlearn.hmm import GaussianHMM

model = GaussianHMM(n_components=3, covariance_type="full", n_iter=1000)
model.fit(X)

GaussianHMM(covariance_type='full', n_components=3, n_iter=1000)

In [78]:
hidden_states = model.predict(X)
df['Regime'] = hidden_states
df['Regime']

,Regime
Date,
2021-03-31,1
2021-04-01,1
2021-04-05,1
2021-04-06,1
2021-04-07,1
...,...
2026-02-23,1
2026-02-24,1
2026-02-25,1


In [79]:
df.columns = [
    col[0] if isinstance(col, tuple) else col
    for col in df.columns
]

In [80]:
print(df.columns)

Index(['Close', 'High', 'Low', 'Open', 'Volume', 'Returns', 'Volatility',
       'Regime'],
      dtype='object')


In [81]:
df.groupby('Regime')[['Returns','Volatility']].mean()
#0->Volatile Bull / Expansion
#1->Stable Bull Regime
#2->Bear / Stress Regime

,Returns,Volatility
Regime,,
0,-0.019114,0.024354
1,0.000035,0.026477
2,0.001244,0.013612


In [82]:
train_size = int(0.8 * len(X_scaled))
X_train = X_scaled[:train_size]
X_test = X_scaled[train_size:]

model.fit(X_train)

print("Train LL:", model.score(X_train))
print("Test LL:", model.score(X_test))

Train LL: -1953.4811226307856
Test LL: -646.512663296946


In [83]:
print("Train LL per sample:", model.score(X_train) / len(X_train))
print("Test LL per sample:", model.score(X_test) / len(X_test))

Train LL per sample: -2.0076887180172514
Test LL per sample: -2.6496420626924015
